# VentriQ — Phase 1: EchoNext Dataset Exploration

**Objective:** Inspect and verify the EchoNext dataset (PhysioNet v1.0.0) linking standard 12-lead ECG waveforms to echocardiography-derived Left Ventricular Ejection Fraction (LVEF) and structural heart disease (SHD) labels.

### Core Tasks Verified:
1. Metadata schema, patient keys, and clinical target definitions.
2. Memory-mapped waveform access with zero-copy loading.
3. Patient-level split isolation (ensuring zero leakage into validation and test sets).
4. Waveform-to-metadata row alignment.
5. Target distributions: continuous LVEF and binary moderate/severe thresholds.
6. Representative 12-lead ECG morphology across LVEF ranges.

In [ ]:
from pathlib import Path
import sys

# Ensure project root is in sys.path
NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.echonext import (
    EchoNextDataset,
    create_mock_echonext_dataset,
    STANDARD_12_LEADS,
    SAMPLING_RATE_HZ,
    RECORDING_DURATION_SEC,
    EXPECTED_TIME_STEPS,
    LVEF_MODERATE_THRESHOLD,
    LVEF_SEVERE_THRESHOLD,
)
from src.utils.paths import get_raw_echonext_dir

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 1. Dataset Directory and Loader Initialization

In [ ]:
data_dir = get_raw_echonext_dir()
print(f"Target EchoNext data directory: {data_dir}")

# If raw data is not yet downloaded, create mock dataset for verification
if not (data_dir / "EchoNext_metadata_100k.csv").exists() and not (data_dir / "echonext_metadata_100k.csv").exists():
    print("Raw dataset not found locally. Generating synthetic mock dataset for Phase 1 verification...")
    create_mock_echonext_dataset(data_dir, num_train=70, num_val=15, num_test=15, seed=42)

# Initialize EchoNext dataset loader
dataset = EchoNextDataset(data_dir)
print(f"Successfully loaded EchoNext dataset with {len(dataset):,} records.")
print(f"Detected patient column: '{dataset.patient_col}'")

## 2. Metadata Schema and First Look

In [ ]:
df = dataset.metadata
print("Metadata DataFrame Shape:", df.shape)
print("\nColumns:")
for i, col in enumerate(df.columns):
    print(f"  {i:2d}: {col:<42} {str(df[col].dtype):<10}")

df.head()

## 3. Patient Isolation and Data Leakage Audit

In [ ]:
isolation_audit = dataset.verify_patient_isolation()
print(f"Is leak-free across splits: {isolation_audit['is_leak_free']}")
print(f"Total overlapping patients: {isolation_audit['total_overlapping_patients']}")
print("\nPairwise Overlaps:")
for pair, count in isolation_audit['overlaps'].items():
    print(f"  {pair}: {count}")

print("\nSplit Record and Patient Breakdown:")
for sp, n_rec in isolation_audit['ecgs_per_split'].items():
    n_pat = isolation_audit['unique_patients_per_split'].get(sp, n_rec)
    print(f"  Split '{sp}': {n_rec:,} ECGs from {n_pat:,} patients (ratio: {n_rec/max(1,n_pat):.2f})")

## 4. Waveform Alignment and Memory-Mapped Slicing

In [ ]:
alignment_res = dataset.verify_waveform_alignment()
print(f"All splits aligned: {alignment_res['all_aligned']}")
for sp, info in alignment_res.get('splits', {}).items():
    print(f"  Split '{sp}': metadata rows={info['metadata_rows']}, waveform rows={info['waveform_rows']}, aligned={info['aligned']}")

# Test memory-mapped single waveform access
sample_ecg = dataset.get_waveform(0)
print(f"\nSample waveform index 0 shape: {sample_ecg.shape} (Time steps x Leads)")
print(f"Dtype: {sample_ecg.dtype}")
print(f"Finite values: {np.isfinite(sample_ecg).all()}")
print(f"Mean: {sample_ecg.mean():.4f}, Std: {sample_ecg.std():.4f}, Min: {sample_ecg.min():.4f}, Max: {sample_ecg.max():.4f}")

## 5. Target Distribution: Left Ventricular Ejection Fraction (LVEF)

In [ ]:
summary = dataset.summarize()
lvef_stats = summary['lvef_statistics']

print("LVEF Descriptive Statistics:")
for k, v in lvef_stats.items():
    print(f"  {k:<24}: {v}")

# Plot LVEF Distribution with Clinical Thresholds
lvef = df["lvef_value"].dropna()
plt.figure(figsize=(9, 4.5), dpi=120)
plt.hist(lvef, bins=35, color="#2b5c8f", edgecolor="#1a3b5c", alpha=0.85, density=True)
plt.axvline(LVEF_SEVERE_THRESHOLD, color="#d9381e", linestyle="--", linewidth=2, label=f"Severe (<= {LVEF_SEVERE_THRESHOLD:.0f}%)")
plt.axvline(LVEF_MODERATE_THRESHOLD, color="#f39c12", linestyle="--", linewidth=2, label=f"Moderate (<= {LVEF_MODERATE_THRESHOLD:.0f}%)")
plt.axvline(50.0, color="#27ae60", linestyle=":", linewidth=1.8, label="Preserved (>= 50%)")
plt.xlabel("Left Ventricular Ejection Fraction (LVEF, %)", fontsize=11)
plt.ylabel("Density", fontsize=11)
plt.title("EchoNext LVEF Distribution with Diagnostic Thresholds", fontsize=12, fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()

## 6. Representative 12-Lead ECG Visualization

In [ ]:
# Find representative examples
red_idx = df.index[df["lvef_value"] <= LVEF_MODERATE_THRESHOLD].tolist()
pres_idx = df.index[df["lvef_value"] >= 55.0].tolist()

i_red = red_idx[0] if red_idx else 0
i_pres = pres_idx[0] if pres_idx else 1

wf_red = dataset.get_waveform(i_red)
wf_pres = dataset.get_waveform(i_pres)
lvef_red = dataset.get_lvef(i_red)
lvef_pres = dataset.get_lvef(i_pres)

time_s = np.linspace(0, RECORDING_DURATION_SEC, EXPECTED_TIME_STEPS, endpoint=False)
window_samples = int(2.5 * SAMPLING_RATE_HZ)  # 2.5-second rhythm slice

fig, axes = plt.subplots(6, 4, figsize=(14, 10), sharex=True, sharey=True, dpi=120)
fig.suptitle(
    f"EchoNext 12-Lead ECG Comparison\n"
    f"Left: Reduced Systolic Function (LVEF = {lvef_red:.1f}%) | Right: Preserved Function (LVEF = {lvef_pres:.1f}%)",
    fontsize=13,
    fontweight="bold"
)

for i, lead in enumerate(STANDARD_12_LEADS):
    r = i % 6
    c_offset = 0 if i < 6 else 1
    # Reduced ECG
    axes[r, c_offset].plot(time_s[:window_samples], wf_red[:window_samples, i], color="#c0392b", lw=1.2)
    axes[r, c_offset].set_ylabel(lead, fontweight="bold", fontsize=9)
    axes[r, c_offset].grid(True, linestyle=":", alpha=0.5)
    # Preserved ECG
    axes[r, c_offset + 2].plot(time_s[:window_samples], wf_pres[:window_samples, i], color="#2980b9", lw=1.2)
    axes[r, c_offset + 2].set_ylabel(lead, fontweight="bold", fontsize=9)
    axes[r, c_offset + 2].grid(True, linestyle=":", alpha=0.5)

for ax in axes[-1, :]:
    ax.set_xlabel("Time (seconds)", fontsize=9)

plt.tight_layout()
plt.show()

## 7. Key Findings and Phase 1 Signoff

1. **Waveform Layout:** Standard 12 leads (`I, II, III, aVR, aVL, aVF, V1-V6`), $f_s = 250$ Hz, 10 seconds (2500 samples).
2. **Storage and Memory Mapping:** Bulk `.npy` format per split efficiently sliced via `np.load(..., mmap_mode='r')`.
3. **Zero Patient Leakage:** Patient keys in official `val` and `test` splits are strictly disjoint from `train`.
4. **Normalization Note:** EchoNext waveforms have already been median filtered and standardized; subsequent rendering must not treat them as uncalibrated raw millivolts.